In [1]:
# Les dépendances sont installées via requirements.txt
# Pour réinstaller : pip install -r ../requirements.txt


# 📢 Étape 7 : Data Storytelling & Communication

Synthétiser les résultats en recommandations actionnables pour un décideur (label, artiste, plateforme) via le framework OIA et des visualisations interactives.

### 1. Préparation de l'environnement

In [2]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath('..'))

print("Librairies prêtes pour la phase de Data Storytelling !")

Librairies prêtes pour la phase de Data Storytelling !


### 2. Synthèse métier — Framework OIA

Le modèle unifié (R²=0.31) révèle que la réputation de l'artiste représente **67% du pouvoir prédictif**, contre 26% pour toutes les features audio combinées. Le framework OIA traduit ces résultats en actions concrètes.

In [3]:
print("=" * 60)
print("RAPPORT EXÉCUTIF — Analyse Spotify")
print("=" * 60)

print("""
RECOMMANDATION PRINCIPALE :
Le succès d'un morceau sur Spotify est prédit à 67% par la
réputation de l'artiste — pas par ses caractéristiques audio.
Investir dans le catalogue et la notoriété artiste avant
d'optimiser la production sonore.
""")

storytelling = {
    "Observation 1": {
        "fait":    "artist_reputation représente 67% de l'importance du modèle unifié (R²=0.31)",
        "insight": "La notoriété préexistante de l'artiste est le signal prédictif dominant",
        "action":  "Prioriser les artistes avec un catalogue établi pour les nouvelles sorties"
    },
    "Observation 2": {
        "fait":    "instrumentalness est la feature audio la plus corrélée à la popularité (r=-0.18)",
        "insight": "Les morceaux sans paroles sous-performent structurellement sur Spotify",
        "action":  "Favoriser les productions vocales pour maximiser les streams"
    },
    "Observation 3": {
        "fait":    "Les anomalies de succès (résidus élevés) concernent surtout les collaborations virales",
        "insight": "Les collaborations multi-artistes génèrent des succès que le modèle ne peut anticiper",
        "action":  "Multiplier les collaborations pour créer des effets de surprise algorithmique"
    }
}

print("\nANALYSE OIA (Observation → Insight → Action)\n")
for titre, contenu in storytelling.items():
    print(f"{'='*55}")
    print(f"  {titre}")
    print(f"  Fait    : {contenu['fait']}")
    print(f"  Insight : {contenu['insight']}")
    print(f"  Action  : {contenu['action']}")

print("\nStorytelling généré avec succès !")

RAPPORT EXÉCUTIF — Analyse Spotify

RECOMMANDATION PRINCIPALE :
Le succès d'un morceau sur Spotify est prédit à 67% par la
réputation de l'artiste — pas par ses caractéristiques audio.
Investir dans le catalogue et la notoriété artiste avant
d'optimiser la production sonore.


ANALYSE OIA (Observation → Insight → Action)

  Observation 1
  Fait    : artist_reputation représente 67% de l'importance du modèle unifié (R²=0.31)
  Insight : La notoriété préexistante de l'artiste est le signal prédictif dominant
  Action  : Prioriser les artistes avec un catalogue établi pour les nouvelles sorties
  Observation 2
  Fait    : instrumentalness est la feature audio la plus corrélée à la popularité (r=-0.18)
  Insight : Les morceaux sans paroles sous-performent structurellement sur Spotify
  Action  : Favoriser les productions vocales pour maximiser les streams
  Observation 3
  Fait    : Les anomalies de succès (résidus élevés) concernent surtout les collaborations virales
  Insight : Les colla

### 3. Visualisations Interactives — Les 4 Insights Clés

Quatre graphiques interactifs qui racontent l'histoire complète : pourquoi l'audio ne suffit pas, ce qui prédit vraiment, et les anomalies que le modèle ne peut pas expliquer.

In [4]:
import joblib
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/processed/cleaned_data_sample.csv')

# Features du modèle unifié
df['artist_reputation'] = df.groupby('artists')['popularity'].transform('mean')
df['pop_rank_in_genre'] = df.groupby('track_genre')['popularity'].transform(lambda x: x.rank(pct=True))

AUDIO = ['danceability','energy','loudness','speechiness',
         'acousticness','instrumentalness','liveness','valence','tempo']

scaler = StandardScaler()
audio_scaled = scaler.fit_transform(df[AUDIO])
audio_scaled_df = pd.DataFrame(audio_scaled, columns=AUDIO, index=df.index)
audio_scaled_df['track_genre'] = df['track_genre'].values
genre_centroids = audio_scaled_df.groupby('track_genre')[AUDIO].mean()
centroid_matrix = np.array([genre_centroids.loc[g].values for g in df['track_genre']])
df['genre_distance'] = np.linalg.norm(audio_scaled - centroid_matrix, axis=1)
df['duration_min'] = df['duration_ms'] / 60000
df['explicit_int'] = df['explicit'].astype(int)

# Charger le modèle et calculer les résidus
try:
    MODEL_FEATURES = AUDIO + ['artist_reputation', 'genre_distance', 'duration_min', 'explicit_int']
    rf = joblib.load('../data/processed/rf_unified_model.pkl')
    df['predicted_rank'] = rf.predict(df[MODEL_FEATURES])
    df['residual'] = df['pop_rank_in_genre'] - df['predicted_rank']
    model_loaded = True
    print(f"Modèle chargé — {len(df):,} morceaux prêts")
except Exception as e:
    model_loaded = False
    print(f"Modèle non disponible : {e}")

Modèle chargé — 97,270 morceaux prêts


In [5]:
# ── GRAPHIQUE 1 : L'audio ne prédit pas — corrélations faibles ──────────────
AUDIO_LABELS = {
    'danceability':'Dansabilité','energy':'Énergie','loudness':'Volume',
    'speechiness':'Paroles','acousticness':'Acoustique',
    'instrumentalness':'Instrumental','liveness':'Live',
    'valence':'Valence','tempo':'Tempo'
}

corr_raw  = df[AUDIO].corrwith(df['popularity']).rename(AUDIO_LABELS).sort_values()
corr_rank = df[AUDIO + ['artist_reputation']].corrwith(df['pop_rank_in_genre'])
corr_rank = corr_rank.rename({**AUDIO_LABELS, 'artist_reputation': 'Réputation artiste ★'}).sort_values()

fig1 = make_subplots(rows=1, cols=2,
    subplot_titles=["Audio → Popularité brute (r < 0.07)",
                    "Audio + Artiste → Rang relatif dans le genre"])

fig1.add_trace(go.Bar(
    x=corr_raw.values, y=corr_raw.index, orientation='h',
    marker_color=['#ff4500' if v < 0 else '#1DB954' for v in corr_raw],
    name='Popularité brute'
), row=1, col=1)

fig1.add_trace(go.Bar(
    x=corr_rank.values, y=corr_rank.index, orientation='h',
    marker_color=['#ff4500' if v < 0 else '#1DB954' for v in corr_rank],
    name='Rang relatif'
), row=1, col=2)

fig1.update_layout(height=420, showlegend=False,
    title_text="Insight 1 : L'audio seul ne prédit pas le succès",
    paper_bgcolor='#0b0c10', plot_bgcolor='#141b22', font_color='#c5c6c7')
fig1.show()

In [6]:
# ── GRAPHIQUE 2 : Réputation artiste vs Rang relatif ────────────────────────
df_sample = df.sample(min(5000, len(df)), random_state=42)
x_vals = df_sample['artist_reputation'].values
y_vals = df_sample['pop_rank_in_genre'].values
m, b   = np.polyfit(x_vals, y_vals, 1)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=x_vals, y=y_vals, mode='markers',
    marker=dict(color='#1DB954', opacity=0.2, size=4),
    hoverinfo='skip', showlegend=False
))
fig2.add_trace(go.Scatter(
    x=[x_vals.min(), x_vals.max()],
    y=[m*x_vals.min()+b, m*x_vals.max()+b],
    mode='lines', line=dict(color='#ff007f', width=2.5),
    name=f'Tendance (r={np.corrcoef(x_vals, y_vals)[0,1]:.2f})'
))

fig2.update_layout(
    title="Insight 2 : La réputation artiste est le signal le plus fort",
    xaxis_title="Réputation artiste (popularité moy. 0-100)",
    yaxis_title="Rang relatif dans le genre (0-1)",
    height=420, paper_bgcolor='#0b0c10', plot_bgcolor='#141b22',
    font_color='#c5c6c7',
    legend=dict(bgcolor='rgba(0,0,0,0)')
)
fig2.show()

In [7]:
# ── GRAPHIQUE 3 : Importance des features du modèle unifié ──────────────────
if model_loaded:
    MODEL_LABELS = {
        'artist_reputation': 'Réputation artiste ★',
        'genre_distance':    'Distance au genre ★',
        'duration_min':      'Durée',
        'explicit_int':      'Contenu explicite',
        'danceability':      'Dansabilité',
        'energy':            'Énergie',
        'loudness':          'Volume',
        'speechiness':       'Paroles',
        'acousticness':      'Acoustique',
        'instrumentalness':  'Instrumental',
        'liveness':          'Live',
        'valence':           'Valence',
        'tempo':             'Tempo',
    }
    imp = pd.DataFrame({
        'feature':    [MODEL_LABELS.get(f, f) for f in MODEL_FEATURES],
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=True)

    fig3 = go.Figure(go.Bar(
        x=imp['importance'], y=imp['feature'], orientation='h',
        marker_color=['#1DB954' if '★' in f else '#45a29e' for f in imp['feature']],
        text=(imp['importance']*100).round(1).astype(str) + '%',
        textposition='outside'
    ))
    fig3.update_layout(
        title="Insight 3 : Ce qui prédit vraiment le succès — Importance des variables",
        height=450, paper_bgcolor='#0b0c10', plot_bgcolor='#141b22',
        font_color='#c5c6c7', xaxis_title="Importance (Gini)", showlegend=False
    )
    fig3.show()
else:
    print("Modèle non disponible pour ce graphique")

In [8]:
# ── GRAPHIQUE 4 : Anomalies — Surprises & Déceptions ────────────────────────
if model_loaded:
    seuil_sup = df['residual'].quantile(0.95)
    seuil_inf = df['residual'].quantile(0.05)

    df_bg   = df[(df['residual'] > seuil_inf) & (df['residual'] < seuil_sup)].sample(3000, random_state=42)
    df_surp = df[df['residual'] >= seuil_sup]
    df_dec  = df[df['residual'] <= seuil_inf]
    top10_s = df.nlargest(10, 'residual')
    top10_d = df.nsmallest(10, 'residual')

    fig4 = go.Figure()
    fig4.add_trace(go.Scatter(
        x=df_bg['predicted_rank'], y=df_bg['pop_rank_in_genre'],
        mode='markers', marker=dict(color='#1f2e3a', size=4, opacity=0.6),
        showlegend=False, hoverinfo='skip'
    ))
    fig4.add_trace(go.Scatter(
        x=[0,1], y=[0,1], mode='lines',
        line=dict(color='rgba(255,255,255,0.2)', dash='dot'),
        showlegend=False, hoverinfo='skip'
    ))
    fig4.add_trace(go.Scatter(
        x=df_surp['predicted_rank'], y=df_surp['pop_rank_in_genre'],
        mode='markers', marker=dict(color='#1DB954', size=7, opacity=0.7),
        name='Surprises', hovertemplate='<b>%{customdata[0]}</b><br>%{customdata[1]}<extra></extra>',
        customdata=np.stack([df_surp['track_name'], df_surp['artists']], axis=-1)
    ))
    fig4.add_trace(go.Scatter(
        x=df_dec['predicted_rank'], y=df_dec['pop_rank_in_genre'],
        mode='markers', marker=dict(color='#ff4500', size=7, opacity=0.7),
        name='Déceptions', hovertemplate='<b>%{customdata[0]}</b><br>%{customdata[1]}<extra></extra>',
        customdata=np.stack([df_dec['track_name'], df_dec['artists']], axis=-1)
    ))
    fig4.add_trace(go.Scatter(
        x=top10_s['predicted_rank'], y=top10_s['pop_rank_in_genre'],
        mode='markers+text', marker=dict(color='#1DB954', size=12, symbol='star'),
        text=top10_s['track_name'].str[:20], textposition='top center',
        textfont=dict(size=8, color='#1DB954'), showlegend=False, hoverinfo='skip'
    ))
    fig4.add_trace(go.Scatter(
        x=top10_d['predicted_rank'], y=top10_d['pop_rank_in_genre'],
        mode='markers+text', marker=dict(color='#ff4500', size=12, symbol='x'),
        text=top10_d['track_name'].str[:20], textposition='bottom center',
        textfont=dict(size=8, color='#ff4500'), showlegend=False, hoverinfo='skip'
    ))
    fig4.update_layout(
        title="Insight 4 : Les anomalies — ce que le modèle ne peut pas expliquer (viralité, marketing)",
        xaxis_title="Rang prédit", yaxis_title="Rang réel",
        height=520, paper_bgcolor='#0b0c10', plot_bgcolor='#080d12',
        font_color='#c5c6c7',
        legend=dict(bgcolor='rgba(0,0,0,0)'),
        annotations=[
            dict(x=0.1, y=0.9, text="Surprises", showarrow=False,
                 font=dict(color='#1DB954', size=12)),
            dict(x=0.9, y=0.1, text="Déceptions", showarrow=False,
                 font=dict(color='#ff4500', size=12))
        ]
    )
    fig4.show()
    print("Visualisations interactives générées !")
else:
    print("Modèle non disponible pour l'analyse des anomalies")

Visualisations interactives générées !
